# File 3B — Final T1 linkage with all audit variables

This notebook preserves the 98-day overlap-day exposure calculation. It carries every generated `t1_*` field, retains `birth_id`, and creates pregnancy-level flags for any overlapping county-quarter containing a finished-water measurement at or above 10 mg/L.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, numpy as np, pandas as pd

BIRTH_PATH = '/content/drive/MyDrive/plos-update-v3/2-birth/births_linkage_T1_v3.csv'
WATER_PATH = '/content/drive/MyDrive/plos-update-v3/1-water/final_county_quarter_complete_1982_1988_v3.csv'
OUTPUT_DIR = '/content/drive/MyDrive/plos-update-v3/3-linked'
os.makedirs(OUTPUT_DIR, exist_ok=True)
births = pd.read_csv(BIRTH_PATH, low_memory=False)
water = pd.read_csv(WATER_PATH, low_memory=False)
print('Births:', f'{len(births):,}', ' Water county-quarters:', f'{len(water):,}')

Mounted at /content/drive
Births: 164,977  Water county-quarters: 2,772


In [ ]:
# Validate and standardize linkage keys.
births['county_fips'] = pd.to_numeric(births['county_fips'], errors='raise').astype(np.int64)
births['t1_start'] = pd.to_datetime(births['t1_start'], errors='raise')
births['t1_end'] = pd.to_datetime(births['t1_end'], errors='raise')
assert births['birth_id'].is_unique
assert ((births['t1_end'] - births['t1_start']).dt.days == 97).all()

water['county_fips'] = pd.to_numeric(water['county_fips'], errors='raise').astype(np.int64)
water['quarter_start'] = pd.to_datetime(water['quarter_start'], errors='raise')
water['quarter_end'] = pd.to_datetime(water['quarter_end'], errors='raise')
water['year'] = water['quarter_start'].dt.year.astype(np.int16)
water['quarter'] = water['quarter_start'].dt.quarter.astype(np.int8)
water['_qindex'] = water['year'].astype(np.int64) * 4 + water['quarter'].astype(np.int64)
for c in ['nitrate_complete', 'nitrate_observed']:
    water[c] = pd.to_numeric(water[c], errors='coerce')
assert water['nitrate_complete'].notna().all()
assert (water['nitrate_complete'] >= 0).all()
assert water[['county_fips','_qindex']].duplicated().sum() == 0

# These audit fields already exist in the V3 water panel. If an older panel lacks one,
# create a transparent fallback rather than silently changing exposure values.
if 'any_pws_ge10' not in water.columns:
    if 'nitrate_max' in water.columns:
        water['any_pws_ge10'] = (pd.to_numeric(water['nitrate_max'], errors='coerce') >= 10).astype('Int64')
    else:
        raise KeyError('Need any_pws_ge10 or nitrate_max to audit any finished-water measure >=10 mg/L.')
if 'county_primary_ge10' not in water.columns:
    water['county_primary_ge10'] = (water['nitrate_observed'] >= 10).fillna(False).astype(np.int8)
if 'nitrate_max' not in water.columns:
    water['nitrate_max'] = np.nan


In [ ]:
# Exact 98-day overlap linkage — same V3 logic, with audit columns carried forward.
def qindex(s):
    return s.dt.year.to_numpy(np.int64) * 4 + ((s.dt.month.to_numpy(np.int64)-1)//3 + 1)

base = births[['birth_id','county_fips','t1_start','t1_end']].rename(columns={'t1_start':'window_start','t1_end':'window_end'}).copy()
start_q = qindex(base['window_start']); end_q = qindex(base['window_end'])
nq = (end_q - start_q + 1).astype(np.int64)
pos = np.repeat(np.arange(len(base), dtype=np.int64), nq)
starts = np.repeat(np.cumsum(nq)-nq, nq)
offset = np.arange(int(nq.sum()), dtype=np.int64) - starts
long = pd.DataFrame({
    'birth_id': base['birth_id'].to_numpy()[pos],
    'county_fips': base['county_fips'].to_numpy(np.int64)[pos],
    'window_start': base['window_start'].to_numpy()[pos],
    'window_end': base['window_end'].to_numpy()[pos],
    '_qindex': start_q[pos] + offset})
wcols = ['county_fips','_qindex','quarter_start','quarter_end','nitrate_complete','nitrate_observed','any_pws_ge10','county_primary_ge10','nitrate_max']
long = long.merge(water[wcols], on=['county_fips','_qindex'], how='left', validate='m:1')
if long['nitrate_complete'].isna().any():
    raise ValueError('Missing completed nitrate during a T1 overlap.')
long['overlap_start'] = long[['window_start','quarter_start']].max(axis=1)
long['overlap_end'] = long[['window_end','quarter_end']].min(axis=1)
long['overlap_days'] = (long['overlap_end'] - long['overlap_start']).dt.days + 1
assert (long['overlap_days'] > 0).all()
obs = long['nitrate_observed'].notna()
long['_complete_num'] = long['nitrate_complete'] * long['overlap_days']
long['_observed_num'] = np.where(obs, long['nitrate_observed'] * long['overlap_days'], 0.0)
long['_obsdays'] = np.where(obs, long['overlap_days'], 0)
long['_impdays'] = np.where(obs, 0, long['overlap_days'])
long['_pws_ge10'] = pd.to_numeric(long['any_pws_ge10'], errors='coerce').fillna(0).gt(0).astype(np.int8)
long['_county_ge10'] = pd.to_numeric(long['county_primary_ge10'], errors='coerce').fillna(0).gt(0).astype(np.int8)


In [ ]:
agg = long.groupby('birth_id', sort=False).agg(
    t1_days=('overlap_days','sum'),
    t1_nquarters=('_qindex','size'),
    _complete_num=('_complete_num','sum'),
    _observed_num=('_observed_num','sum'),
    t1_obsdays=('_obsdays','sum'),
    t1_impdays=('_impdays','sum'),
    t1_any_pws_ge10=('_pws_ge10','max'),
    t1_any_county_primary_ge10=('_county_ge10','max'),
    t1_max_finished_water=('nitrate_max','max')
).reset_index()
agg['t1_mean_complete'] = agg['_complete_num'] / agg['t1_days']
agg['t1_mean_observed'] = np.where(agg['t1_obsdays']>0, agg['_observed_num']/agg['t1_obsdays'], np.nan)
agg['t1_obsfrac'] = agg['t1_obsdays']/agg['t1_days']
agg['t1_anyimp'] = (agg['t1_impdays']>0).astype(np.int8)
agg['t1_allobs'] = (agg['t1_impdays']==0).astype(np.int8)
# Canonical manuscript exclusion flag: any underlying finished-water system measure >=10
# in any county-quarter overlapping T1. This is intentionally conservative.
agg['t1_any_ge10'] = agg['t1_any_pws_ge10'].astype(np.int8)
assert agg['t1_days'].eq(98).all()
agg = agg.drop(columns=['_complete_num','_observed_num'])

linked = births.merge(agg, on='birth_id', how='left', validate='1:1')
assert linked['t1_mean_complete'].notna().all()

# Keep analysis variables plus EVERY T1 variable already present or generated.
core = ['birth_id','county_fips','birth_year','gest_age_weeks','conception_quarter','birthweight_g','infant_male','maternal_age','maternal_race_broad','married','prenatal_by5','live_birth_order','maternal_education_years']
t1cols = sorted([c for c in linked.columns if c.lower().startswith('t1_')])
keep = core + [c for c in t1cols if c not in core]
linked_all = linked[keep].copy()
print('Retained T1 fields:')
print(t1cols)


Retained T1 fields:
['t1_allobs', 't1_any_county_primary_ge10', 't1_any_ge10', 't1_any_pws_ge10', 't1_anyimp', 't1_days', 't1_end', 't1_impdays', 't1_max_finished_water', 't1_mean_complete', 't1_mean_observed', 't1_nquarters', 't1_obsdays', 't1_obsfrac', 't1_start']


In [ ]:
# QA needed for manuscript finalization. These are linkage-cohort diagnostics;
# the Stata file repeats them after final analytic restrictions.
qa = pd.Series({
    'n_births': len(linked_all),
    'pct_any_imputed_T1': 100*linked_all['t1_anyimp'].mean(),
    'pct_all_98_days_observed': 100*linked_all['t1_allobs'].mean(),
    'mean_fraction_T1_days_observed': linked_all['t1_obsfrac'].mean(),
    'pct_any_PWS_measure_ge10': 100*linked_all['t1_any_pws_ge10'].mean(),
    'pct_any_county_primary_ge10': 100*linked_all['t1_any_county_primary_ge10'].mean(),
    'mean_t1_mean_complete': linked_all['t1_mean_complete'].mean(),
    'mean_t1_mean_observed': linked_all['t1_mean_observed'].mean()
})
display(qa.to_frame('value'))
for n in [1,31,61,91,98]:
    print(f'T1 observed days >= {n}:', int((linked_all.t1_obsdays>=n).sum()))

OUT_CSV = os.path.join(OUTPUT_DIR, 'birth_water_linked_T1_allvars_v3.csv')
OUT_PARQUET = os.path.join(OUTPUT_DIR, 'birth_water_linked_T1_allvars_v3.parquet')
linked_all.to_csv(OUT_CSV, index=False)
linked_all.to_parquet(OUT_PARQUET, index=False)
qa.to_frame('value').to_csv(os.path.join(OUTPUT_DIR,'t1_linkage_final_audit.csv'))
print('Saved:', OUT_PARQUET)

,value
n_births,164977.000000
pct_any_imputed_T1,32.475436
pct_all_98_days_observed,67.524564
mean_fraction_T1_days_observed,0.779176
pct_any_PWS_measure_ge10,14.031047
pct_any_county_primary_ge10,6.973093
mean_t1_mean_complete,3.467369
mean_t1_mean_observed,3.564306


T1 observed days >= 1: 145922
T1 observed days >= 31: 135307
T1 observed days >= 61: 124198
T1 observed days >= 91: 113335
T1 observed days >= 98: 111400
Saved: /content/drive/MyDrive/plos-update-v3/3-linked/birth_water_linked_T1_allvars_v3.parquet
